# 위험도 분류 모델 학습

In [1]:
import os
import json
import math
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report
from typing import List, Dict, Any

import ast

c:\Users\silve\miniconda3\envs\313_20251008\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
LABEL_ORDER = ["positive", "danger", "critical", "emergency"]

# tqdm.pandas()를 호출하여 progress_apply를 활성화합니다.
tqdm.pandas(desc="Parsing list-like columns")

In [3]:
def load_and_parse_csv(path: str) -> pd.DataFrame:
    """
    CSV를 로드하고, CSV 저장으로 인해 문자열로 변환된 리스트 형태의 컬럼들을
    ast.literal_eval을 사용하여 다시 파이썬 객체(리스트)로 파싱합니다.

    Args:
        path (str): 로드할 CSV 파일 경로.

    Returns:
        pd.DataFrame: 리스트 컬럼이 파싱된 DataFrame.
    """
    df = pd.read_csv(path)
    # CSV에 리스트 형태로 저장된 컬럼 목록
    list_columns = ['input_ids', 'attention_mask', 'seq_texts', 'seq_delta_t', 'seq_hours', 'seq_emo_vectors']
    for col in list_columns:
        if col in df.columns:
            # progress_apply를 사용하여 파싱 진행 상황을 시각적으로 보여줍니다.
            df[col] = df[col].progress_apply(ast.literal_eval)
    return df

In [4]:
class ContextDataset(Dataset):
    """
    전처리된 데이터를 모델 학습에 사용할 수 있는 형태로 변환하는 PyTorch Dataset 클래스.
    텍스트 데이터 외에 시간, 감정, 문맥 기반의 추가 특성을 생성합니다.
    """
    def __init__(self, df: pd.DataFrame, label_map: Dict[str, int]):
        """
        Args:
            df (pd.DataFrame): 전처리 및 파싱이 완료된 DataFrame.
            label_map (Dict[str, int]): 레이블 문자열을 정수 인덱스로 매핑하는 딕셔너리.
        """
        self.df = df
        self.label_map = label_map
        # 감정 특성 관련 컬럼 이름을 미리 추출하여 사용합니다.
        self.emo_cols = [c for c in df.columns if c.startswith("emo_")]
        # 문맥 위험도 계산에 사용할 감정 점수 컬럼의 인덱스를 미리 찾아둡니다.
        self.emo_score_indices = {
            'emergency': self.emo_cols.index('emo_emergency_score') if 'emo_emergency_score' in self.emo_cols else None,
            'critical': self.emo_cols.index('emo_critical_score') if 'emo_critical_score' in self.emo_cols else None,
            'danger': self.emo_cols.index('emo_danger_score') if 'emo_danger_score' in self.emo_cols else None,
        }

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        """
        하나의 데이터 샘플(발화)에 대한 모델 입력값을 생성합니다.
        
        Args:
            idx (int): 가져올 데이터의 인덱스.

        Returns:
            Dict[str, Any]: 모델 입력으로 사용될 텐서 딕셔너리.
        """
        row = self.df.iloc[idx]
        
        # 1. 토크나이징된 텍스트 데이터
        input_ids = row["input_ids"]
        attention_mask = row["attention_mask"]

        # 2. 시간 관련 특성
        last_hour = row["hour"]
        # 시간(hour)을 순환적인 특성으로 변환하여 23시와 0시가 가깝다는 것을 표현
        hour_sin = math.sin(2 * math.pi * last_hour / 24)
        hour_cos = math.cos(2 * math.pi * last_hour / 24)
        
        # 3. 감정 어휘 기반 특성
        emo_vec = row[self.emo_cols].values.astype(np.float32)

        # 4. 문맥 기반 위험도 특성 (Contextual Risk Feature)
        # 이전 대화들의 위험도와 시간 경과를 함께 고려한 특성입니다.
        # 최근에 위험한 발화가 많았을수록 높은 값을 가집니다.
        seq_emo_vectors = row["seq_emo_vectors"]
        seq_delta_t = row["seq_delta_t"]
        
        weighted_context_risk = 0.0
        # 문맥에 2개 이상의 발화가 있을 때만 계산 (현재 발화 제외)
        if len(seq_emo_vectors) > 1:
            # 현재 발화를 제외한 이전 발화들에 대해 반복
            for i in range(len(seq_emo_vectors) - 1):
                emo_vec_context = seq_emo_vectors[i]
                delta_t = seq_delta_t[i+1]  # 해당 발화와 다음 발화 사이의 시간 간격
                
                # 각 위험도 레벨의 감정 점수에 가중치를 부여하여 합산
                utterance_risk_score = 0
                if self.emo_score_indices['emergency'] is not None: utterance_risk_score += emo_vec_context[self.emo_score_indices['emergency']] * 3.0
                if self.emo_score_indices['critical'] is not None: utterance_risk_score += emo_vec_context[self.emo_score_indices['critical']] * 2.0
                if self.emo_score_indices['danger'] is not None: utterance_risk_score += emo_vec_context[self.emo_score_indices['danger']] * 1.0
                
                # 위험 점수가 0보다 클 경우, 시간 경과(delta_t)에 따라 지수적으로 점수를 감쇠시킴
                if utterance_risk_score > 0:
                    # lambda는 감쇠율을 조절하는 하이퍼파라미터. 최근 발화일수록 더 큰 영향을 줌.
                    # 10분(600초)이 지나면 영향력이 약 10%로 감소(90% 감소)하는 수준입니다. exp(-0.00384 * 600) ~= 0.1
                    decay_lambda = 0.00384
                    weighted_context_risk += utterance_risk_score * math.exp(-decay_lambda * delta_t)
        
        # 최종 문맥 위험도 점수에 log1p를 적용하여 값의 범위를 안정화
        context_risk_feat = math.log1p(weighted_context_risk)

        # 모델에 입력될 최종 딕셔너리 구성
        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "time_feats": torch.tensor([hour_sin, hour_cos], dtype=torch.float),
            "emo_feats": torch.tensor(emo_vec, dtype=torch.float),
            "context_risk_feats": torch.tensor([context_risk_feat], dtype=torch.float),
        }
        # 레이블이 있는 경우 (학습/검증 데이터)
        if "label" in row.index and not pd.isna(row["label"]):
            item["label"] = torch.tensor(self.label_map.get(row["label"], -1), dtype=torch.long)

        return item

In [5]:
def collate_fn(batch: List[Dict[str, Any]], pad_token_id: int) -> Dict[str, Any]:
    """
    DataLoader에서 생성된 샘플 리스트를 미니배치(mini-batch)로 구성합니다.
    가변 길이의 시퀀스(input_ids)를 패딩하여 동일한 길이로 만듭니다.
    """
    input_ids = [b["input_ids"] for b in batch]
    attention_mask = [b["attention_mask"] for b in batch]
    
    # `pad_sequence`를 사용하여 배치 내 최대 길이에 맞춰 패딩을 동적으로 적용
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    attention_mask_padded = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)

    # 나머지 특성들은 텐서로 변환 후 쌓아줍니다 (stack).
    time_feats = torch.stack([b["time_feats"] for b in batch], dim=0)
    emo_feats = torch.stack([b["emo_feats"] for b in batch], dim=0)
    context_risk_feats = torch.stack([b["context_risk_feats"] for b in batch], dim=0)

    out = {
        "input_ids": input_ids_padded,
        "attention_mask": attention_mask_padded,
        "time_feats": time_feats,
        "emo_feats": emo_feats,
        "context_risk_feats": context_risk_feats,
    }
    if "label" in batch[0]:
        out["labels"] = torch.stack([b["label"] for b in batch], dim=0)
    return out

In [6]:
class ContextRiskModel(nn.Module):
    """
    문맥을 고려한 위험도 분류 모델.
    사전 학습된 언어 모델(Encoder)과 LSTM, 추가 특성을 결합한 하이브리드 구조.
    """
    def __init__(self, encoder_name: str, emo_feat_dim: int, time_feat_dim: int = 2, num_labels: int = 4, lstm_hidden_size: int = 256, context_risk_feat_dim: int = 1, use_attention: bool = True):
        super().__init__()
        # 모델의 설정을 저장하여 나중에 모델을 불러올 때 동일한 구조를 재현할 수 있도록 함
        self.config = {
            "encoder_name": encoder_name, "emo_feat_dim": emo_feat_dim, "time_feat_dim": time_feat_dim,
            "num_labels": num_labels, "lstm_hidden_size": lstm_hidden_size, 
            "context_risk_feat_dim": context_risk_feat_dim, "use_attention": use_attention,
        }
        self.use_attention = use_attention
        self.encoder = AutoModel.from_pretrained(encoder_name)
        enc_dim = self.encoder.config.hidden_size
        
        # 양방향 LSTM: 텍스트 시퀀스의 순방향 및 역방향 문맥을 모두 학습
        self.lstm = nn.LSTM(input_size=enc_dim, hidden_size=lstm_hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        
        if self.use_attention:
            # Multi-head Attention: LSTM 출력의 여러 부분에 가중치를 부여하여 중요한 정보를 강조
            self.attention = nn.MultiheadAttention(embed_dim=lstm_hidden_size * 2, num_heads=8, batch_first=True)
            self.attention_norm = nn.LayerNorm(lstm_hidden_size * 2) # 잔차 연결을 위한 Layer Normalization
            pooled_dim = lstm_hidden_size * 2
        else:
            # Attention을 사용하지 않을 경우, LSTM의 마지막 은닉 상태를 사용
            pooled_dim = lstm_hidden_size * 2
        
        # 1. 문맥 위험도를 제외한 특성들로 1차 분류기를 구성합니다.
        # (언어 모델 출력 차원) + (시간 특성 차원) + (감정 특성 차원)
        base_input_dim = pooled_dim + time_feat_dim + emo_feat_dim
        self.base_classifier = nn.Sequential(
            nn.Linear(base_input_dim, 512), nn.ReLU(), nn.Dropout(0.2), nn.Linear(512, num_labels)
        )

        # 2. 문맥 위험도를 '위험도 편향(Risk Bias)'으로 변환하는 작은 네트워크를 추가합니다.
        # 이 네트워크는 positive 점수는 낮추고(-), 나머지 위험도 점수는 높이도록(+) 학습됩니다.
        self.risk_bias_generator = nn.Sequential(
            nn.Linear(context_risk_feat_dim, 16),
            nn.ReLU(),
            nn.Linear(16, num_labels)
        )

    def forward(self, input_ids, attention_mask, time_feats, emo_feats, context_risk_feats):
        # 1. 언어 모델(Encoder)을 통과시켜 토큰별 임베딩(hidden states)을 얻음
        sequence_output = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        
        # 2. LSTM에 입력하기 전, 패딩을 무시하도록 시퀀스를 압축 (성능 및 효율성 향상)
        lengths = attention_mask.sum(dim=1).long().cpu()
        packed_input = pack_padded_sequence(sequence_output, lengths, batch_first=True, enforce_sorted=False)
        packed_out, (h_n, c_n) = self.lstm(packed_input)
        lstm_output, _ = pad_packed_sequence(packed_out, batch_first=True) # 다시 패딩된 형태로 복원
        
        if self.use_attention:
            # 3a. Attention 적용 및 풀링
            attn_output, _ = self.attention(lstm_output, lstm_output, lstm_output, key_padding_mask=attention_mask == 0)
            # 잔차 연결(Residual Connection) 및 정규화
            pooled = self.attention_norm(lstm_output + attn_output)
            # 어텐션 마스크를 고려하여 평균 풀링 수행
            pooled = self._masked_mean_pooling(pooled, attention_mask)
        else:
            # 3b. Attention 미사용 시, LSTM의 마지막 은닉 상태를 결합하여 사용
            pooled = torch.cat((h_n[-2,:,:], h_n[-1,:,:]), dim=1)
        
        # 4. 1차 분류: 문맥 위험도를 제외한 특성들로 기본 로짓(logits)을 계산
        base_features = torch.cat([pooled, time_feats, emo_feats], dim=-1)
        base_logits = self.base_classifier(base_features)

        # 5. 위험도 편향(Risk Bias) 계산
        # context_risk_feats가 클수록 이 편향 값의 절대값이 커지도록 학습됩니다.
        risk_bias = self.risk_bias_generator(context_risk_feats)

        # 6. 최종 로짓 = 기본 로짓 + 위험도 편향. 문맥 위험도가 높을수록 위험 클래스 점수가 가산됩니다.
        return base_logits + risk_bias
    
    def _masked_mean_pooling(self, hidden_states, attention_mask):
        """어텐션 마스크를 고려하여 패딩 토큰을 제외하고 평균 풀링을 수행합니다."""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9) # 0으로 나누는 것을 방지
        return sum_embeddings / sum_mask

    def save_pretrained(self, save_directory):
        """모델의 가중치와 설정을 저장합니다."""
        os.makedirs(save_directory, exist_ok=True)
        json.dump(self.config, open(os.path.join(save_directory, "config.json"), 'w'), indent=4)
        torch.save(self.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))

    @classmethod
    def from_pretrained(cls, load_directory):
        """저장된 가중치와 설정으로부터 모델을 불러옵니다."""
        config = json.load(open(os.path.join(load_directory, "config.json"), 'r'))
        model = cls(**config)
        model.load_state_dict(torch.load(os.path.join(load_directory, "pytorch_model.bin"), map_location=torch.device('cpu')))
        return model

In [7]:
class FocalLoss(nn.Module):
    """
    Focal Loss: 클래스 불균형 문제를 해결하기 위한 손실 함수.
    맞추기 쉬운 샘플(easy example)의 손실은 줄이고, 맞추기 어려운 샘플(hard example)의 손실에 더 집중합니다.
    """
    def __init__(self, alpha: List[float] = None, gamma: float = 2.0, reduction: str = 'mean'):
        """
        Args:
            alpha (List[float], optional): 각 클래스에 대한 가중치. 클래스 불균형이 심할 때 사용.
            gamma (float, optional): Focusing 파라미터. 높을수록 쉬운 샘플의 영향력을 줄임.
            reduction (str, optional): 손실 집계 방식 ('mean', 'sum', 'none').
        """
        super().__init__()
        self.alpha = torch.tensor(alpha) if alpha is not None else None
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # 표준 CrossEntropyLoss 계산
        BCE_loss = F.cross_entropy(inputs, targets, reduction='none')
        # pt는 모델이 정답을 맞출 확률
        pt = torch.exp(-BCE_loss)
        # Focal Loss 계산: (1-pt)^gamma * BCE_loss
        F_loss = (1-pt)**self.gamma * BCE_loss
        
        # alpha 가중치가 주어지면, 해당 클래스의 손실에 가중치를 적용
        if self.alpha is not None:
            self.alpha = self.alpha.to(inputs.device)
            F_loss = self.alpha[targets] * F_loss
        if self.reduction == 'mean': return torch.mean(F_loss)
        elif self.reduction == 'sum': return torch.sum(F_loss)
        else: return F_loss

In [8]:
# 스크립트 실행을 위한 arguments 설정
class Arguments:
    def __init__(self):
        self.train_preprocessed_path = "../../data/label/preprocessed_train_data.csv" # 전처리된 학습 데이터 파일 경로 (CSV)
        self.val_preprocessed_path = "../../data/label/preprocessed_val_data.csv"     # 전처리된 검증 데이터 파일 경로 (CSV)
        self.output_dir = "../../model/label"                                         # 학습된 모델이 저장될 디렉토리
        self.tokenizer_name = "klue/roberta-base"                                     # 사전 학습된 토크나이저 이름
        self.encoder_name = "klue/roberta-base"                                       # 사전 학습된 인코더 모델 이름
        self.epochs = 50                                                              # 총 학습 에폭 수
        self.batch_size = 128                                                         # 배치 크기
        self.learning_rate = 2e-5                                                     # 학습률
        self.lstm_hidden_size = 256                                                   # LSTM 은닉층 크기
        self.num_workers = 0                                                          # DataLoader를 위한 워커 수
        self.early_stopping_patience = 15                                             # 조기 중단을 위한 patience 값
        self.use_amp = True                                                           # Automatic Mixed Precision 사용 여부
        self.force_cpu = False                                                        # CUDA 사용 가능 시에도 CPU 강제 사용
        self.use_attention = True                                                     # 모델에 어텐션 메커니즘 사용 여부

args = Arguments()

In [9]:
"""
모델 학습 파이프라인 전체를 실행합니다.
"""

device = torch.device("cuda" if torch.cuda.is_available() and not args.force_cpu else "cpu")
print(f"Starting training on device: {device}")

print(f"Loading and parsing data from {args.train_preprocessed_path}...")
df = load_and_parse_csv(args.train_preprocessed_path)
# 유효하지 않은 레이블을 가진 데이터를 필터링
if "label" in df.columns:
    original_len = len(df)
    df = df[df['label'].isin(LABEL_ORDER)].copy()
    if len(df) < original_len: print(f"Filtered out {original_len - len(df)} rows with invalid labels from training data.")

train_df = df

print(f"Loading and parsing validation data from {args.val_preprocessed_path}...")
val_df = load_and_parse_csv(args.val_preprocessed_path)
if "label" in val_df.columns:
    original_len = len(val_df)
    val_df = val_df[val_df['label'].isin(LABEL_ORDER)].copy()
    if len(val_df) < original_len: print(f"Filtered out {original_len - len(val_df)} rows with invalid labels from validation data.")

tokenizer = AutoTokenizer.from_pretrained(args.tokenizer_name, use_fast=True)
label_map = {label: i for i, label in enumerate(LABEL_ORDER)}

# --- Focal Loss의 alpha 값 계산 로직 ---
# 목표: 데이터가 적은 클래스(불균형)와 위험도가 높은 클래스에 더 높은 가중치를 부여
# 1. 클래스별 데이터 수의 역빈도(Inverse Frequency)를 기반으로 가중치 계산
class_counts = train_df['label'].value_counts().reindex(LABEL_ORDER).fillna(0)
total_samples = len(train_df)
num_classes = len(LABEL_ORDER)
inverse_freq_weights = [total_samples / (num_classes * count) if count > 0 else 0.0 for count in class_counts]

# 2. 위험도에 따른 수동 가중치 부여
# 이 값들을 조정하여 특정 위험 클래스에 대한 민감도를 제어할 수 있습니다.
risk_level_weights = [1.0, 4.0, 8.0, 12.0]

# 3. 두 가중치를 곱하여 최종 alpha 값 생성
final_alpha_weights = [inv_freq * risk_weight for inv_freq, risk_weight in zip(inverse_freq_weights, risk_level_weights)]
print(f"Using FocalLoss with final alpha weights: {final_alpha_weights}")

loss_fct = FocalLoss(alpha=final_alpha_weights, gamma=2.0, reduction='mean').to(device)

train_dataset = ContextDataset(train_df, label_map)
val_dataset = ContextDataset(val_df, label_map)

pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, collate_fn=lambda b: collate_fn(b, pad_token_id), num_workers=args.num_workers, pin_memory=device.type == 'cuda')
val_loader = DataLoader(val_dataset, batch_size=args.batch_size, shuffle=False, collate_fn=lambda b: collate_fn(b, pad_token_id), num_workers=args.num_workers, pin_memory=device.type == 'cuda')

emo_dim = sum(1 for c in df.columns if c.startswith("emo_"))
model = ContextRiskModel(
    encoder_name=args.encoder_name, emo_feat_dim=emo_dim, num_labels=len(LABEL_ORDER),
    lstm_hidden_size=args.lstm_hidden_size, use_attention=args.use_attention
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=args.learning_rate)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(len(train_loader) * args.epochs * 0.1), num_training_steps=len(train_loader) * args.epochs)

# AMP(Automatic Mixed Precision) 사용 시, 그래디언트 스케일러 초기화
use_amp = args.use_amp and device.type == 'cuda'
scaler = torch.amp.GradScaler(enabled=use_amp)

best_f1_score = float('-inf') # F1-score는 높을수록 좋으므로 초기값을 음의 무한대로 설정
patience_counter = 0

for epoch in range(args.epochs):
    model.train()
    total_train_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{args.epochs} | Training"):
        optimizer.zero_grad()

        labels = batch.pop("labels").to(device)
        inputs = {k: v.to(device) for k, v in batch.items()}
        
        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**inputs)
            loss = loss_fct(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer) # clip_grad_norm_ 전에 unscale 필요
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        scheduler.step()
        total_train_loss += loss.item()

    # --- 검증 단계 ---
    model.eval()
    total_eval_loss = 0
    all_preds = []
    all_labels = []
    print(f"Epoch {epoch+1} | Average Training Loss: {total_train_loss / len(train_loader):.4f}")
    for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{args.epochs} | Validation"):
        with torch.no_grad():
            labels = batch.pop("labels").to(device)
            inputs = {k: v.to(device) for k, v in batch.items()}
            
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(**inputs)
                loss = loss_fct(logits, labels)

            total_eval_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_val_loss = total_eval_loss / len(val_loader)
    print(f"Epoch {epoch+1} | Validation Loss: {avg_val_loss:.4f}")

    # --- Classification Report 및 혼동 행렬 출력 ---
    target_names = [label for label, i in sorted(label_map.items(), key=lambda item: item[1])]
    report = classification_report(all_labels, all_preds, target_names=target_names, output_dict=True, zero_division=0)
    macro_avg_f1 = report['macro avg']['f1-score']
    print(f"Epoch {epoch+1} | Validation Macro Avg F1-score: {macro_avg_f1:.4f}")
    print("--- Validation Classification Report ---")
    print(classification_report(all_labels, all_preds, target_names=target_names, digits=4, zero_division=0))

    # --- 조기 종료(Early Stopping) 및 모델 저장 (Macro Avg F1-score 기준) ---
    if macro_avg_f1 > best_f1_score:
        best_f1_score = macro_avg_f1
        patience_counter = 0
        print(f"New best model found based on Macro Avg F1-score! Saving to {args.output_dir}")
        model.save_pretrained(args.output_dir)
        tokenizer.save_pretrained(args.output_dir)
    else:
        patience_counter += 1
        print(f"Macro Avg F1-score did not improve. Patience: {patience_counter}/{args.early_stopping_patience}")
    
    if patience_counter >= args.early_stopping_patience:
        print("Early stopping triggered.")
        break

print(f"Training complete. Best model saved with Macro Avg F1-score: {best_f1_score:.4f}")

Starting training on device: cuda
Loading and parsing data from ../../data/label/preprocessed_train_data.csv...


Parsing list-like columns: 100%|██████████| 168476/168476 [00:21<00:00, 7958.37it/s] 


Loading and parsing validation data from ../../data/label/preprocessed_val_data.csv...


Parsing list-like columns: 100%|██████████| 32973/32973 [00:04<00:00, 7353.86it/s]


Using FocalLoss with final alpha weights: [0.8650974592807115, 3.990053050397878, 9.04083713442447, 12.543193944658146]


...

Epoch 49 | Validation Loss: 0.0128
Epoch 49 | Validation Macro Avg F1-score: 0.8509
--- Validation Classification Report ---
              precision    recall  f1-score   support

    positive     0.9995    0.9920    0.9957     32111
      danger     0.7723    0.9564    0.8546       642
    critical     0.6780    0.9267    0.7831       150
   emergency     0.6442    0.9571    0.7701        70

    accuracy                         0.9909     32973
   macro avg     0.7735    0.9580    0.8509     32973
weighted avg     0.9929    0.9909    0.9916     32973

Macro Avg F1-score did not improve. Patience: 14/15


Epoch 50/50 | Training: 100%|██████████| 1317/1317 [12:39<00:00,  1.74it/s]


Epoch 50 | Average Training Loss: 0.0196


Epoch 50/50 | Validation: 100%|██████████| 258/258 [01:08<00:00,  3.79it/s]


Epoch 50 | Validation Loss: 0.0128
Epoch 50 | Validation Macro Avg F1-score: 0.8511
--- Validation Classification Report ---
              precision    recall  f1-score   support

    positive     0.9995    0.9920    0.9957     32111
      danger     0.7704    0.9564    0.8534       642
    critical     0.6814    0.9267    0.7853       150
   emergency     0.6442    0.9571    0.7701        70

    accuracy                         0.9909     32973
   macro avg     0.7739    0.9580    0.8511     32973
weighted avg     0.9929    0.9909    0.9915     32973

Macro Avg F1-score did not improve. Patience: 15/15
Early stopping triggered.
Training complete. Best model saved with Macro Avg F1-score: 0.8513
